In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
models = [
    ("Blackbox Direct", "blackbox-direct"),
    ("SupProto Direct", "labsup-proto-direct"),
    ("SupProto AudioSet (Probe)", "labsup-proto-audioset-rila"),
    ("SupProto AudioSet",         "labsup-proto-audioset-rila-ft"),
    ("ProtoSSL AudioSet (Probe)", "protossl-audioset-pila"),
    ("ProtoSSL AudioSet",         "protossl-audioset-pila-ft"),
]

datasets = [
    # dataset         fold    path
    ("IEMOCAP",       5,    "/opt/gpu_working/steven/protossl-audio/runs-iemocap-fold{}"),
    ("ESC-50",        5,    "/opt/gpu_working/steven/protossl-audio/runs-esc50-fold{}"),
    ("UrbanSound8K",  10,   "/opt/gpu_working/steven/protossl-audio/runs-us8k-fold{}"),
    ("VoxCeleb1 ID",  None, "/opt/gpu_working/steven/protossl-audio/runs-voxceleb"),
    ("SpeechCmds V2", None, "/opt/gpu_working/steven/protossl-audio/runs-speechcmds"),
]

In [ ]:
data = []
for ds_name, n_folds, fpath in datasets:
    for model_name, model_dir in models:
        if n_folds is not None:
            accs = []
            los = []
            his = []
            for i in range(n_folds):
                fold_path = Path(fpath.format(i))
                metrics = pd.read_csv(fold_path / model_dir / "metrics-bootstrapped.csv", index_col="Label")
                accs.append(metrics.loc["Multiclass", "Accuracy"])
                los.append(metrics.loc["Multiclass", "Accuracy 95% CI (lo)"])
                his.append(metrics.loc["Multiclass", "Accuracy 95% CI (hi)"])
            acc = np.mean(accs)
            lo = np.mean(los)
            hi = np.mean(his)
        else:
            metrics = pd.read_csv(Path(fpath) / model_dir / "metrics-bootstrapped.csv", index_col="Label")
            acc = metrics.loc["Multiclass", "Accuracy"]
            lo = metrics.loc["Multiclass", "Accuracy 95% CI (lo)"]
            hi = metrics.loc["Multiclass", "Accuracy 95% CI (hi)"]
        data.append({
            "Dataset": ds_name,
            "Model": model_name,
            "Accuracy": acc,
            "95% CI Lo": lo,
            "95% CI Hi": hi,
        })
results = pd.DataFrame(data)
results["value"] = (
    results["Accuracy"].apply(lambda x: f"{x:0.3f}")
    + " ["
    + results["95% CI Lo"].apply(lambda x: f"{x:0.3f}")
    + "-"
    + results["95% CI Hi"].apply(lambda x: f"{x:0.3f}")
    + "]"
)
results = results[["Model", "Dataset", "value"]].pivot(columns="Model", index="Dataset", values="value")
results.columns.name = None

In [ ]:
results

In [ ]:
print(
    results.loc[
        ["UrbanSound8K", "IEMOCAP", "VoxCeleb1 ID"],
        ["ProtoSSL AudioSet", "SupProto AudioSet", "ProtoSSL AudioSet (Probe)", "SupProto AudioSet (Probe)", "SupProto Direct"]
    ].to_latex()
)